# M3 Agentic AI - Transformando funções em ferramentas

## 1. Introdução

### 1.1. Visão geral do laboratório

Neste laboratório não avaliado, você criará um conjunto de ferramentas com `aisuite` para fornecer a um LLM. Você verá como o LLM solicita o uso de ferramentas e também como o LLM escolhe certas ferramentas quando relevantes à sua tarefa.

### 🎯 1.2 Resultado de aprendizagem

Aplicar padrões de design de chamada de ferramentas em fluxos de trabalho de agentes.

Para conseguir isso, você dará aos LLMs acesso controlado a funções Python via AISuite, gerenciará a passagem de parâmetros e o fluxo de execução, e validará saídas de várias etapas geradas pela orquestração de ferramentas.


## 2. Configuração: Inicializar ambiente e cliente

Como nos laboratórios anteriores, você começará inicializando seu ambiente. Você importará vários pacotes agora e também mais tarde, à medida que cria ferramentas para seu LLM.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../..'))
import json
from utils import display_functions
from dotenv import load_dotenv
_ = load_dotenv()

### 2.1 Começando com AISuite

Em seguida, você inicializará o cliente **AISuite** que você viu nas lições anteriores. Uma vez inicializado, esta será sua interface para gerar respostas do agente e chamar ferramentas.

Execute a célula abaixo para inicializar o cliente.

In [ ]:
import aisuite as ai

# Create an instance of the AISuite client
client = ai.Client()

## 3. Construa sua primeira ferramenta

### 3.1 Definindo sua função

Agora que você configurou seu ambiente, é hora de criar sua primeira ferramenta. Execute a célula abaixo para definir uma função que retorna a hora atual como uma string. Observe que esta ferramenta inclui uma docstring explicando o propósito da função. Isso é importante para o `aisuite` porque ele usará isso para ajudar a definir a ferramenta para o LLM.

In [ ]:
from datetime import datetime

def get_current_time():
    """
    Retorna a hora atual como uma string.
    """
    return datetime.now().strftime("%H:%M:%S")

Teste sua função para ver exatamente o que ela retorna.

<div style="background-color:#ffe4e1; padding:12px; border-radius:6px; color:black;">
  <strong>Nota:</strong> A plataforma DeepLearning.AI usa o Horário de Greenwich (GMT) por padrão. Se você executar esta função localmente, ela retornará seu horário local.
</div>

In [ ]:
get_current_time()

Ótimo! Exatamente como esperado, a função retorna uma string com sua hora atual.

### 3.2 Transformando sua função em uma ferramenta LLM

Agora, vamos usar o `aisuite` para passar esta ferramenta para um LLM e obter uma resposta. Para configurar sua ferramenta, primeiro você configura a `response` (resposta) do LLM. Criar uma resposta primeiro requer criar a estrutura da mensagem. A estrutura da mensagem inclui o prompt que o usuário pede, bem como um dicionário que representa o histórico da conversa e cada mensagem tendo um `role` (papel) (por exemplo, "user", "assistant", "system") e `content` (conteúdo).

In [ ]:
# Message structure
prompt = "What time is it?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

Após definir a estrutura, você pode construir sua conclusão de chat. Isso fará a chamada ao LLM e retornará o resultado. Vamos dar uma olhada nos parâmetros desta chamada.
* `model`: O modelo que será usado
* `messages`: A lista de mensagens passadas para o LLM
* `tools`: A lista de ferramentas às quais o LLM tem acesso
* `max_turns`: Esta é a quantidade máxima de mensagens que o LLM poderá fazer. Isso pode ajudar a evitar que o LLM entre em loops infinitos e chame repetidamente uma ferramenta.

Execute a célula abaixo para chamar o LLM e ver a resposta.

In [ ]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=[get_current_time],
    max_turns=5
)

# See the LLM response
print(response.choices[0].message.content)

E assim, você deu ao seu LLM acesso a ferramentas! A ferramenta `aisuite` transformou sua função em uma ferramenta que aumentou o conhecimento do LLM sobre o mundo.

### 3.3 Olhando mais de perto a resposta
Embora o conteúdo da resposta final tenha sido exatamente o esperado, há muita coisa acontecendo nos bastidores da `response`. Vamos usar uma função `utility` útil para olhar mais de perto. `pretty_print_chat_completion` extrairá as etapas da resposta e mostrará as partes importantes em um formato fácil de ler.

In [ ]:
display_functions.pretty_print_chat_completion(response)

Como você pode ver, o LLM enviou uma mensagem para usar `get_current_time`. Isso foi executado em sua máquina e enviado de volta ao LLM. Finalmente, o LLM, tendo o histórico completo da conversa, usou essa informação para dar a resposta final. O `aisuite` lidou com todas as complexidades de extrair a mensagem com a chamada da ferramenta, executá-la localmente e, em seguida, passar a mensagem de volta para o LLM quando você usou `max_turns` e passou o nome da função para o cliente.

### 3.4 Definindo ferramentas manualmente

Você viu que a docstring fornecida por nossa ferramenta ajudou o `aisuite` a transformar automaticamente sua função em uma ferramenta para o LLM. Isso é muito conveniente. Mas o que está acontecendo nos bastidores para pegar sua função e torná-la uma ferramenta?

Na verdade, o que a ferramenta parece para o LLM é mais complicado. Vejamos como o LLM *realmente* recebe uma ferramenta abaixo. Como antes, as ferramentas são dadas em uma lista. Mas nessa lista, as ferramentas têm um esquema definido que esperam. Dentro desse esquema há várias partes importantes:
* `name`: O nome da função correspondente que você definiu localmente
* `description`: Uma descrição que explica o que a função faz e é usada pelo LLM para ajudá-lo a decidir quando usá-la
* `parameters`: Se sua função tiver parâmetros, eles também seriam descritos com o nome do parâmetro e uma descrição do que o parâmetro deve ser.

Execute a célula abaixo para definir sua ferramenta usando o esquema.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time", # <--- Your functions name
        "description": "Retorna a hora atual como uma string.", # <--- a description for the LLM
        "parameters": {}
    }
}]

Neste caso, onde você define um esquema, o `aisuite` espera que você lide com a execução. Portanto, você não usará `max_turns` e, em vez disso, lidará com a execução você mesmo. Vamos configurar isso definindo a resposta.

In [ ]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=tools, # <-- Your list of tools with get_current_time
    # max_turns=5 # <-- When defining tools manually, you must handle calls yourself and cannot use max_turns
)

Agora você pode dar uma olhada na resposta do LLM.

In [ ]:
print(json.dumps(response.model_dump(), indent=2, default=str))

Observe que na `response` você pode ver `tool_calls` em `message`. Esta resposta do LLM está dizendo que o LLM agora quer chamar uma ferramenta, especificamente, `get_current_time`. Você pode adicionar alguma lógica para lidar com esta situação. Em seguida, passe isso de volta para o modelo e obtenha a resposta final.

Execute a célula abaixo para rodar a função localmente, retorná-la ao LLM e receber a resposta final.

In [ ]:
response2 = None

# Create a condition in case tool_calls is in response object
if response.choices[0].message.tool_calls:
    # Pull out the specific tool metadata from the response
    tool_call = response.choices[0].message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)

    # Run the tool locally
    tool_result = get_current_time()

    # Append the result to the messages list
    messages.append(response.choices[0].message)
    messages.append({
        "role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)
    })

    # Send the list of messages with the newly appended results back to the LLM
    response2 = client.chat.completions.create(
        model="openai:gpt-4o",
        messages=messages,
        tools=tools,
    )

    print(response2.choices[0].message.content)


Você agora implementou seu próprio manuseio manual de chamadas de ferramentas LLM. Você pode optar por ter as ferramentas fornecidas automaticamente ao seu LLM com `max_turns` ou escrever o esquema e lidar com as partes intermediárias manualmente.

## 4. Dando ao LLM mais ferramentas

Agora que você explorou como as ferramentas são criadas e processadas localmente, vamos criar mais algumas ferramentas.

### 4.1 Três novas ferramentas

Você definirá três novas ferramentas para seu LLM:

- **Weather Tool (`get_weather_from_ip`)**
  Detecta a localização do usuário e retorna a temperatura atual, máxima e mínima usando chamadas de API externas para detectar seu endereço IP e enviá-lo para uma API de clima.

- **File Writing Tool (`write_txt_file`)**
  Cria um arquivo de texto com o conteúdo especificado em seu ambiente local. A função recebe dois argumentos, `file_path` e `content`.

- **QR Code Generator (`generate_qr_code`)**
  Gera uma imagem de código QR a partir de dados, com incorporação de imagem opcional. A função recebe três argumentos: `data`, `filename` e `img_path`.

Execute a célula abaixo para importar alguns novos pacotes e definir as ferramentas.

In [ ]:
import requests
import qrcode
from qrcode.image.styledpil import StyledPilImage


def get_weather_from_ip():
    """
    Gets the current, high, and low temperature in Fahrenheit for the user's
    location and returns it to the user.
    """
    # Get location coordinates from the IP address
    lat, lon = requests.get('https://ipinfo.io/json').json()['loc'].split(',')

    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "fahrenheit",
        "timezone": "auto"
    }

    # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    # Format and return the simplified string
    return (
        f"Current: {weather_data['current']['temperature_2m']}°F, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]}°F, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]}°F"
    )

# Write a text file
def write_txt_file(file_path: str, content: str):
    """
    Escreve uma string em um arquivo .txt (sobrescreve se existir).
    Args:
        file_path (str): Destination path.
        content (str): Text to write.
    Returns:
        str: Path to the written file.
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path


# Create a QR code
def generate_qr_code(data: str, filename: str, image_path: str):
    """Generate a QR code image given data and an image path.

    Args:
        data: Text or URL to encode
        filename: Name for the output PNG file (without extension)
        image_path: Path to the image to be used in the QR code
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)

    img = qr.make_image(image_factory=StyledPilImage, embedded_image_path=image_path)
    output_file = f"{filename}.png"
    img.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."

### 4.2 Usando suas novas ferramentas

Agora é hora de usar suas novas ferramentas! A `response` parecerá quase a mesma, mas diferente de antes, você passará todas as ferramentas para o LLM. O LLM escolherá a ferramenta apropriada com base no prompt que você enviar. Vamos começar com a ferramenta `get_weather_from_ip`.


In [ ]:
prompt = "Can you get the weather for my location?"

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

Você pode ver que o LLM escolheu apropriadamente a ferramenta correta com base na intenção do prompt, mesmo tendo acesso a outras ferramentas.

Agora, execute a célula abaixo para pedir ao LLM que crie uma nota para você.

In [ ]:
prompt = "Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?"

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

E agora você tem um arquivo de texto com seu lembrete. Você pode ver que o LLM passou os argumentos corretos para a ferramenta, a ferramenta rodou localmente e então imprimiu uma resposta de que a ação foi concluída. Você pode até abrir o arquivo de texto e ler o conteúdo para ter certeza de que ele agora existe.

In [ ]:
with open('reminders.txt', 'r') as file:
    contents = file.read()
    print(contents)

Finalmente, vamos usar esta ferramenta de geração de código QR para fazer um código QR legal que leva os usuários ao site DeepLearning.AI.

Execute a célula abaixo para criar o código QR.

In [ ]:
prompt = "Can you make a QR code for me using my company's logo that goes to www.deeplearning.ai? The logo is located at `dl_logo.jpg`. You can call it dl_qr_code."

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

O LLM também passou com sucesso os argumentos corretos para a função e, em seguida, essa informação foi usada para executar sua função e criar o código QR. Dê uma olhada no QR executando a célula abaixo e experimente para ver se funciona levando você ao site DeepLearning.AI.

In [ ]:
from IPython.display import Image, display

# Display image directly
Image('dl_qr_code.png')

### 4.3 Usando várias ferramentas de uma vez

Por último, é importante lembrar que os LLMs podem usar várias ferramentas e passar por uma sequência de chamadas de ferramentas para realizar várias coisas. Vamos usar suas ferramentas e examinar a `response` para ver o que acontece. Você usará um prompt complexo,

> Can you help me create a qr code that goes to www.deeplearning.com from the image dl_logo.jpg? Also write me a txt note with the current weather please.

Este prompt requer uma boa quantidade de lógica para entender o que chamar e quando. Por exemplo, embora você peça para escrever uma nota de texto primeiro e depois descreva o conteúdo, o LLM precisará dessa informação primeiro para passá-la para a nota de texto. Se o LLM pedisse para usar `write_txt_file` primeiro, ele não teria as informações de `get_weather_from_ip`. Este é um bom exemplo do poder dos LLMs de analisar a linguagem natural e usar ferramentas apropriadas na ordem certa para realizar uma ampla variedade de tarefas.

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">

<h4>🔍 O que observar:</h4>

<ul>
  <li>O LLM <b>escolhe automaticamente</b> qual ferramenta usar com base na solicitação do usuário</li>
  <li><b>Parâmetros são inferidos</b> da mensagem do usuário (como nomes de arquivos, conteúdo, URLs)</li>
  <li>Cada ferramenta <b>retorna informações</b> que o LLM incorpora em sua resposta</li>
  <li><b>Ferramentas sem parâmetros</b> como clima e hora são perfeitas para solicitações rápidas de informações</li>
  <li>A conversa parece <b>natural</b> apesar de operações complexas acontecerem nos bastidores</li>
</ul>

</div>

In [ ]:
prompt = "Can you help me create a qr code that goes to www.deeplearning.com from the image dl_logo.jpg? Also write me a txt note with the current weather please."

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        get_weather_from_ip,
        get_current_time,
        write_txt_file,
        generate_qr_code
    ],
    max_turns=10
)

display_functions.pretty_print_chat_completion(response)

E assim, você vê na sequência de ferramentas que ele chamou as ferramentas na ordem correta para realizar a tarefa, mas respondeu na ordem em que você solicitou.

### Opções de Modelo

Você pode experimentar diferentes modelos OpenAI ao executar esses fluxos de trabalho de chamada de ferramentas. Cada modelo oferece um equilíbrio diferente de capacidade, custo e velocidade:

- **`openai:gpt-4o`** — otimizado para raciocínio e velocidade
- **`openai:gpt-4.1`** — forte desempenho de raciocínio, bom para tarefas complexas
- **`openai:gpt-4.1-mini`** — mais leve, mais rápido e mais barato que o GPT-4.1 completo
- **`openai:gpt-3.5-turbo`** — eficiente para tarefas mais simples e iterações rápidas

Escolher um modelo depende de seus objetivos:
- Use modelos menores para prototipagem rápida e de baixo custo
- Mude para modelos mais fortes quando as tarefas exigirem melhor raciocínio ou orquestração de várias etapas


### Principais Conclusões

- O tool calling (chamada de ferramentas) permite que os LLMs vão além da geração de texto — eles agora podem usar funções como parte de seu raciocínio.
- Funções claras e bem documentadas (com docstrings precisas) ajudam o modelo a saber quando e como usar cada ferramenta.
- O AISuite lida com a complexidade de traduzir funções python em esquemas de ferramentas e orquestrar fluxos de trabalho de várias etapas.
- Escolher o modelo certo importa: modelos menores são mais rápidos e baratos para tarefas simples, enquanto modelos mais fortes são melhores para fluxos de trabalho pesados em raciocínio.
- Observar o fluxo da conversa (prompts, chamadas de ferramentas, resultados, resposta final) é essencial para depurar e melhorar comportamentos agênticos.

Com esses elementos no lugar, agora você tem a base para projetar agentes que combinam o raciocínio do LLM com ferramentas externas para concluir tarefas mais complexas.


<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Parabéns!</strong>

Você completou o laboratório sobre **transformar funções em ferramentas** com AISuite.  
Ao longo do caminho, **você** expôs funções Python como ferramentas, deixou o LLM escolhê-las e chamá-las, e inspecionou a orquestração de ferramentas em várias etapas.

Com essas habilidades, **você** pode projetar fluxos de trabalho agênticos que combinam o raciocínio do LLM com ações reais — confiáveis, auditáveis e fáceis de estender. 🌟

</div>
